# Select 100 cases for clinical validation

Created by: Sahana Kowshik

Date: 12/2025

In [1]:
import pandas as pd
import numpy as np

In [2]:
# test = pd.read_csv("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/testing_data_grpo/test_summary.csv")

In [3]:
# # Add ETPR variables
# def add_etpr_labels(row):
#     value = row['NACCETPR']
    
#     # NC
#     if value == 88:
#         row['ETPR'] = "NC"
    
#     # AD
#     elif value == 1:
#         row['ETPR'] = 'AD'

#     # LBD
#     elif value == 2:
#         row['ETPR'] = 'LBD'

#     # VD
#     elif value == 8:
#         row['ETPR'] = 'VD'

#     # FTLD and its variants, including CBD and PSP, with or without ALS
#     elif value in [4, 5, 6, 7]:
#         row['ETPR'] = 'FTLD'
    
#     # SEF: seizure, epilepsy, etc.
#     elif value in [17, 18, 23, 26, 27, 28, 29]:
#         row['ETPR'] = 'SEF'

#     # Psychiatric conditions
#     elif value in [19, 20, 21, 22, 24, 25]:
#         row['ETPR'] = 'PSY'

#     # Other degenerative
#     elif value not in [30, 88, 99]:
#         row['ETPR'] = 'ODE'

#     else:
#         row['ETPR'] = np.NaN

#     return row


# test = test.apply(add_etpr_labels, axis=1)

In [4]:
# print(len(test))
# print(len(test['NACCID'].unique()))

In [5]:
# test_etpr = pd.read_csv("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/testing_data_grpo/with_summary/test_etpr.csv")
# test_cog = pd.read_csv("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/testing_data_grpo/with_summary/test_cog.csv")
nacc_multi = pd.read_csv("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/testing_data_grpo/with_summary/nacc_multilabel.csv")

In [6]:
len(nacc_multi)

31029

In [7]:
nacc_multi['ID'] = nacc_multi['NACCID']
nacc_multi.drop(['NACCID'], inplace=True, axis=1)

In [8]:
nacc_multi.head()

,AD,FTLD,LBD,ODE,PSY,SEF,VD,diagnosis,options,question,visit_summary,ID
0,0,0,0,0,0,0,0,NC,"A. Not applicable, no cognitive impairment.\nB...","Using the information provided, identify all t...",The subject is an 83-year-old right-handed mal...,NACC282885
1,0,1,0,0,0,0,0,DE,"A. Systemic and External Factors (infectious, ...","Using the information provided, identify all t...","The subject is a 64-year-old right-handed, Whi...",NACC837281
2,1,0,0,0,0,0,0,MCI,"A. Other Dementia Etiology, including MSA, Dow...",Determine all the etiologies underlying the pa...,"The subject is a 77-year-old right-handed, Eng...",NACC355961
3,0,0,0,0,0,0,0,NC,A. Lewy Body Disease (including Dementia with ...,Determine all the etiologies underlying the pa...,The patient is a 57-year-old White female who ...,NACC385116
4,0,1,0,0,0,0,0,DE,A. Alzheimer's Disease\nB. Other Dementia Etio...,"Using the information provided, identify all t...",The subject is a 70-year-old male of White rac...,NACC260495


In [10]:
# test['NACCUDSD'].value_counts()

In [9]:
nacc_multi['diagnosis'].value_counts()

diagnosis
NC     13819
DE     12541
MCI     4669
Name: count, dtype: int64

In [11]:
cols = ['AD', 'FTLD', 'LBD', 'ODE', 'PSY', 'SEF', 'VD']
multi_cols_nonzero = nacc_multi[cols].astype(bool).sum(axis=1)

nacc_multi[multi_cols_nonzero == 2].head()
print(len(nacc_multi[multi_cols_nonzero > 1]))

4507


In [13]:
for i, row in nacc_multi.iterrows():
    labels = [col for col in cols if row[col] == 1]
    label_2 = [col for col in cols if row[col] == 2]
    if (len(labels) == 0) & (len(label_2) != 0):
        raise ValueError
    if (len(labels) == 0) & (len(label_2) == 0) & (row['diagnosis'] != "NC"):
        raise ValueError

In [14]:
# Add a new column 'single_label' based on which column in cols has value 1
def get_single_label(row):
    if row['diagnosis'] == "NC":
        return "NC"
    labels = [col for col in cols if row[col] == 1]
    if len(labels) == 1:
        return labels[0]
    elif len(labels) == 0:
        return np.nan
    else:
        raise ValueError(f"More than one label found for row {row.name}: {labels}")

nacc_multi['ETPR'] = nacc_multi.apply(get_single_label, axis=1)

In [17]:
nacc_multi[nacc_multi['ETPR'].isna()]

,AD,FTLD,LBD,ODE,PSY,SEF,VD,diagnosis,options,question,visit_summary,ID,ETPR


In [15]:
nacc_multi['ETPR'].value_counts()

ETPR
NC      13819
AD      13099
ODE      1410
FTLD     1258
LBD       595
VD        403
SEF       240
PSY       205
Name: count, dtype: int64

In [16]:
# test['ETPR'].value_counts()

In [18]:
len(nacc_multi)

31029

In [19]:
n_cases = 42
random_state = 42

In [20]:
cog_cases = nacc_multi[nacc_multi['ETPR'] == 'NC'].sample(n=16, random_state=random_state).reset_index(drop=True)

In [21]:
# Select 40 cases with NACCUDSD == 3 and equal number from each ETPR category
mci_subset = nacc_multi[nacc_multi['diagnosis'] == "MCI"]

# Get the ETPR categories present
etpr_categories = ['AD', 'FTLD', 'LBD', 'ODE', 'PSY', 'SEF', 'VD']
n_per_category = n_cases // len(etpr_categories)

mci_cases = (
    mci_subset
    .groupby('ETPR', group_keys=False)
    .apply(lambda x: x.sample(n=min(len(x), n_per_category), random_state=random_state))
    .sample(n=n_cases, random_state=random_state)
    .sample(frac=1, random_state=random_state)  # Shuffle all rows
    .reset_index(drop=True)
)

/scratch/ipykernel_3024320/2753343096.py:11: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=min(len(x), n_per_category), random_state=random_state))


In [22]:
len(mci_cases)

42

In [23]:
mci_cases['ETPR'].value_counts()

ETPR
SEF     6
FTLD    6
ODE     6
AD      6
LBD     6
VD      6
PSY     6
Name: count, dtype: int64

In [24]:
# Select 40 cases with NACCUDSD == 4 and equal number from each ETPR category
de_subset = nacc_multi[nacc_multi['diagnosis'] == "DE"]

# Get the ETPR categories present
etpr_categories = ['AD', 'FTLD', 'LBD', 'ODE', 'PSY', 'SEF', 'VD']
n_per_category = n_cases // len(etpr_categories)

# Sample equal number from each ETPR category
de_cases = (
    de_subset
    # .loc[
    #     (~de_subset['NACCID'].isin(cog_cases['NACCID']))
    # ]
    .groupby('ETPR', group_keys=False)  # Ensure balanced sampling by amyloid status
    .apply(lambda x: x.sample(n=min(len(x), n_per_category), random_state=random_state))
    .sample(n=n_cases, random_state=random_state)
    .sample(frac=1, random_state=random_state)  # Shuffle all rows
    .reset_index(drop=True)
)

/scratch/ipykernel_3024320/3757741887.py:15: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=min(len(x), n_per_category), random_state=random_state))


In [25]:
len(de_cases)

42

In [26]:
len(cog_cases) + len(mci_cases) + len(de_cases)

100

In [27]:
len(set(mci_cases['ID']).union(set(de_cases['ID'])).union(set(cog_cases['ID'])))

100

In [28]:
# cog_cases_with_questions = test_cog[test_cog['ID'].isin(cog_cases['NACCID'])].reset_index(drop=True)
# etpr_cases_with_questions = nacc_multi[(nacc_multi['ID'].isin(mci_cases['NACCID'])) | (nacc_multi['ID'].isin(de_cases['NACCID']))].reset_index(drop=True)
etpr_cases_with_questions = pd.concat([cog_cases, mci_cases, de_cases], axis=0).sample(frac=1, random_state=random_state).reset_index(drop=True)

In [32]:
# Create a mapping from expanded descriptions in map_dict to ETIOLOGY_OPTION_TEXTS (first phrase, "Alzheimer's Disease" to "Alzheimer's disease (AD)", etc.)
map_full_to_etio_text = {
    "Alzheimer's Disease": "Alzheimer's disease (AD)",
    "Lewy Body Disease (including Dementia with Lewy Bodies and Parkinson's disease).": "Lewy body disease (LBD)",
    "Frontotemporal Dementia (FTLD. Including CBD, PSP, with/without ALS).": "Frontotemporal lobar degeneration and its variants, including primary progressive aphasia, corticobasal degeneration and progressive supranuclear palsy, and with or without amyotrophic lateral sclerosis (FTLD)",
    "Vascular Dementia.": "Vascular brain injury or vascular dementia including stroke (VD)",
    "Systemic and External Factors (infectious, metabolic, substance abuse, etc.).": "Systemic and environmental factors including infectious diseases (HIV included), metabolic, substance abuse / alcohol, medications, systemic disease and delirium (SEF)",
    "Psychiatric Conditions (depression, bipolar, schizophrenia, anxiety, PTSD).": "Psychiatric conditions including schizophrenia, depression, bipolar disorder, anxiety and posttraumatic stress disorder (PSY)",
    "Other Dementia Etiology, including MSA, Down's syndrome, Huntington's, Epilepsy, Neoplasm, and any etiology not appearing in another option.": "Other (Multiple system atrophy, Essential tremor, Down syndrome, Huntington's disease, Prion disease, Traumatic brain injury, Normal-pressure hydrocephalus, Epilepsy, CNS neoplasm, etc)",
    "Not applicable, no cognitive impairment.": "Not applicable (no cognitive impairment)",
}

In [33]:
etpr_cases_with_questions_modified = etpr_cases_with_questions.copy()

# ensure options are strings and do substring replacements (regex=False to avoid regex metacharacters)
etpr_cases_with_questions_modified['options'] = etpr_cases_with_questions_modified['options'].astype(str)
for old, new in map_full_to_etio_text.items():
    etpr_cases_with_questions_modified['options'] = (
        etpr_cases_with_questions_modified['options']
        .str.replace(old, new, regex=False)
    )

In [34]:
etpr_cases_with_questions_modified.head()

,AD,FTLD,LBD,ODE,PSY,SEF,VD,diagnosis,options,question,visit_summary,ID,ETPR
0,0,0,0,0,0,0,1,DE,A. Frontotemporal lobar degeneration and its v...,Determine all the etiologies underlying the pa...,The patient is a 62-year-old male of White rac...,NACC538507,VD
1,0,0,0,0,1,0,0,MCI,A. Psychiatric conditions including schizophre...,Determine all the etiologies underlying the pa...,The subject is a 76-year-old male of Asian des...,NACC239592,PSY
2,2,0,0,0,0,0,1,DE,"A. Other (Multiple system atrophy, Essential t...",Determine all the etiologies underlying the pa...,"The subject is a 95-year-old white, right-hand...",NACC580806,VD
3,0,0,0,0,0,0,1,MCI,A. Not applicable (no cognitive impairment)\nB...,"Based on the clinical details, determine the m...","The patient is an 81-year-old right-handed, En...",NACC830390,VD
4,0,0,0,1,0,0,0,MCI,A. Frontotemporal lobar degeneration and its v...,"Using the information provided, identify all t...",The patient is a 76-year-old right-handed fema...,NACC156088,ODE


In [35]:
print(etpr_cases_with_questions.iloc[0]['options'])
print("\n")
print(etpr_cases_with_questions_modified.iloc[0]['options'])

A. Frontotemporal Dementia (FTLD. Including CBD, PSP, with/without ALS).
B. Not applicable, no cognitive impairment.
C. Psychiatric Conditions (depression, bipolar, schizophrenia, anxiety, PTSD).
D. Other Dementia Etiology, including MSA, Down's syndrome, Huntington's, Epilepsy, Neoplasm, and any etiology not appearing in another option.
E. Vascular Dementia.
F. Systemic and External Factors (infectious, metabolic, substance abuse, etc.).
G. Lewy Body Disease (including Dementia with Lewy Bodies and Parkinson's disease).
H. Alzheimer's Disease


A. Frontotemporal lobar degeneration and its variants, including primary progressive aphasia, corticobasal degeneration and progressive supranuclear palsy, and with or without amyotrophic lateral sclerosis (FTLD)
B. Not applicable (no cognitive impairment)
C. Psychiatric conditions including schizophrenia, depression, bipolar disorder, anxiety and posttraumatic stress disorder (PSY)
D. Other (Multiple system atrophy, Essential tremor, Down synd

In [101]:
etpr_cases_with_questions_modified.to_csv("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/clinician_validation.csv", index=False)

In [36]:
print(etpr_cases_with_questions_modified.iloc[0]['question'])

Determine all the etiologies underlying the patient's cognitive impairment based on the provided information, if applicable.
